In [ ]:
# @title 1. Build Edge Environment (Run this first)
import os

print("➔ 1. Cloning Repository...")
!git clone https://github.com/DhakshanaS/Thermal-Edge-AI.git
%cd Thermal-Edge-AI

print("➔ 2. Installing System Dependencies (Driver-Safe Mode)...")
!wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
!dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
!apt-get update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get install -yq --no-install-recommends tzdata libnvinfer-dev libnvinfer-bin > /dev/null
!pip install ultralytics > /dev/null

print("➔ 3. Generating ONNX Model & Building TensorRT Engine (This takes ~45 seconds)...")
from ultralytics import YOLO
YOLO('yolov8n.pt').export(format='onnx')
!mv yolov8n.onnx models/best.onnx
!TRTEXEC=$(find /usr -name trtexec -type f | grep -v "site-packages" | head -n 1) && $TRTEXEC --onnx=models/best.onnx --saveEngine=models/thermal_cpp.engine > /dev/null

print("➔ 4. Validating Deployment Path via CMake...")
cpp_code = """#include <iostream>
#include <fstream>
#include <vector>
#include <NvInfer.h>
#include <cuda_runtime_api.h>
using namespace nvinfer1;
class Logger : public ILogger {
    void log(Severity severity, const char* msg) noexcept override {
        if (severity <= Severity::kWARNING) { std::cout << "[TRT] " << msg << std::endl; }
    }
} gLogger;
int main() {
    std::cout << "--- Starting Edge Device Pipeline ---" << std::endl;
    std::ifstream file("models/thermal_cpp.engine", std::ios::binary);
    if (!file.good()) { std::cerr << "Error: Cannot find engine!" << std::endl; return -1; }
    file.seekg(0, file.end);
    size_t size = file.tellg();
    file.seekg(0, file.beg);
    std::vector<char> engineData(size);
    file.read(engineData.data(), size);
    file.close();
    IRuntime* runtime = createInferRuntime(gLogger);
    ICudaEngine* engine = runtime->deserializeCudaEngine(engineData.data(), size);
    if (!engine) { std::cerr << "Failed to create TensorRT Engine." << std::endl; return -1; }
    IExecutionContext* context = engine->createExecutionContext();
    const int INPUT_SIZE = 1 * 3 * 640 * 640;
    const int OUTPUT_SIZE = 1 * 7 * 8400;
    void* buffers[2];
    cudaMalloc(&buffers[0], INPUT_SIZE * sizeof(float));
    cudaMalloc(&buffers[1], OUTPUT_SIZE * sizeof(float));
    std::cout << "CUDA GPU Memory allocated successfully." << std::endl;
    cudaFree(buffers[0]);
    cudaFree(buffers[1]);
    delete context;
    delete engine;
    delete runtime;
    std::cout << "C++ Engine shut down safely." << std::endl;
    return 0;
}
"""
with open('src/main.cpp', 'w') as f:
    f.write(cpp_code)

os.makedirs('build', exist_ok=True)
%cd build
!cmake ../src > /dev/null
!make > /dev/null
print("\n➔ 5. Running C++ Memory Validation:")
!./thermal_engine
%cd ..

print("\n✅ Edge Environment Successfully Built!")

In [ ]:
# @title 2. Launch Live Real-Time Demo
print("➔ Installing Python dependencies...")
!pip install Flask pyngrok ultralytics opencv-python-headless tensorrt > /dev/null
from pyngrok import ngrok
import os

app_py_code = """import base64
import cv2
import numpy as np
from flask import Flask, render_template, request, jsonify
from ultralytics import YOLO

app = Flask(__name__)

# Note: Loading a raw trtexec engine via Ultralytics relies on a UnicodeDecodeError fallback 
# in their loader. For full production, use native TensorRT Python bindings.
model = YOLO('models/thermal_cpp.engine', task='detect')

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/process_frame', methods=['POST'])
def process_frame():
    data = request.json['image']
    encoded = data.split(",", 1)[1]
    frame = cv2.imdecode(np.frombuffer(base64.b64decode(encoded), np.uint8), cv2.IMREAD_COLOR)
    
    annotated = model(frame, verbose=False)[0].plot()
    b64_img = base64.b64encode(cv2.imencode('.jpg', annotated)[1]).decode('utf-8')
    return jsonify({'annotated_image': b64_img})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""
with open('web/app.py', 'w') as f:
    f.write(app_py_code)

print("Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken")
NGROK_TOKEN = input("Paste your ngrok Auth Token here: ")
!ngrok config add-authtoken {NGROK_TOKEN} > /dev/null

ngrok.kill()
public_url = ngrok.connect(5000)

print("\n" + "="*70)
print(f"🚀 LIVE C++ EDGE DEMO READY!")
print(f"🔗 CLICK HERE: {public_url}")
print("="*70 + "\n")

%cd /content/Thermal-Edge-AI
!python web/app.py